In [5]:
!git clone https://github.com/abachaa/MedQuAD.git

Cloning into 'MedQuAD'...
remote: Enumerating objects: 11310, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 11310 (delta 7), reused 4 (delta 4), pack-reused 11300 (from 1)
Receiving objects: 100% (11310/11310), 11.01 MiB | 17.13 MiB/s, done.
Resolving deltas: 100% (6807/6807), done.


In [6]:
# List the contents of the downloaded folder
!ls MedQuAD

10_MPlus_ADAM_QA	     6_NINDS_QA
11_MPlusDrugs_QA	     7_SeniorHealth_QA
12_MPlusHerbsSupplements_QA  8_NHLBI_QA_XML
1_CancerGov_QA		     9_CDC_QA
2_GARD_QA		     LICENSE.txt
3_GHR_QA		     QA-TestSet-LiveQA-Med-Qrels-2479-Answers.zip
4_MPlus_Health_Topics_QA     readme.txt
5_NIDDK_QA


In [7]:
import glob
import xml.etree.ElementTree as ET
import pandas as pd

# 1. Get a list of all XML files in the MedQuAD subfolders
file_paths = glob.glob('/content/MedQuAD/*/*.xml')

dataset = []

# 2. Loop through every file and extract the data
for file in file_paths:
    try:
        tree = ET.parse(file)
        root = tree.getroot()

        # Get the focus area (e.g., the disease or drug name)
        focus = root.find('Focus').text if root.find('Focus') is not None else ''

        # Go through all the Question-Answer pairs in the file
        qa_pairs = root.find('QAPairs')
        if qa_pairs is not None:
            for qa in qa_pairs.findall('QAPair'):
                question = qa.find('Question')
                answer = qa.find('Answer')

                # Extract text and question type (intent)
                q_text = question.text if question is not None else ''
                q_type = question.attrib.get('qtype', '') if question is not None else ''
                a_text = answer.text if answer is not None else ''

                if q_text and a_text:
                    dataset.append({
                        'intent': q_type,
                        'focus_area': focus,
                        'query': q_text,
                        'response': a_text
                    })
    except Exception as e:
        print(f"Error parsing {file}: {e}")

# 3. Convert the extracted data into a Pandas DataFrame
df = pd.DataFrame(dataset)

# Show the first 5 rows to verify it worked
print(f"Total Q&A pairs extracted: {len(df)}")
df.head()

Total Q&A pairs extracted: 16407


,intent,focus_area,query,response
0,information,Prenatal Testing,Do you have information about Prenatal Testing,Summary : Prenatal testing provides informatio...
1,information,Liver Function Tests,Do you have information about Liver Function T...,Summary : Your liver helps your body digest fo...
2,information,Bird Flu,What is (are) Bird Flu ?,"Birds, just like people, get the flu. Bird flu..."
3,information,Corneal Disorders,What is (are) Corneal Disorders ?,Your cornea is the outermost layer of your eye...
4,information,Food Allergy,What is (are) Food Allergy ?,Food allergy is an abnormal response to a food...


In [8]:
# Save to CSV in your Colab environment
df.to_csv('/content/medquad_clean.csv', index=False)
print("Saved as medquad_clean.csv!")

Saved as medquad_clean.csv!


In [9]:
import glob
import xml.etree.ElementTree as ET
import pandas as pd

# 1. Get a list of all XML files in the MedQuAD subfolders
file_paths = glob.glob('/content/MedQuAD/*/*.xml')

dataset = []

# 2. Loop through every file and extract the data
for file in file_paths:
    try:
        tree = ET.parse(file)
        root = tree.getroot()

        focus = root.find('Focus').text if root.find('Focus') is not None else ''

        qa_pairs = root.find('QAPairs')
        if qa_pairs is not None:
            for qa in qa_pairs.findall('QAPair'):
                question = qa.find('Question')
                answer = qa.find('Answer')

                q_text = question.text if question is not None else ''
                q_type = question.attrib.get('qtype', '') if question is not None else ''
                a_text = answer.text if answer is not None else ''

                if q_text and a_text:
                    dataset.append({
                        'intent': q_type,
                        'focus_area': focus,
                        'query': q_text,
                        'response': a_text
                    })
    except Exception as e:
        print(f"Error parsing {file}: {e}")

# 3. Convert the extracted data into a Pandas DataFrame
df = pd.DataFrame(dataset)
print(f"Total Q&A pairs extracted: {len(df)}")

Total Q&A pairs extracted: 16407


In [10]:
from sklearn.model_selection import train_test_split

# 1. Drop rows with missing values
df_clean = df.dropna(subset=['query', 'response', 'intent']).copy()

# 2. Filter out extremely rare intents (keep intents with at least 20 examples)
intent_counts = df_clean['intent'].value_counts()
valid_intents = intent_counts[intent_counts >= 20].index
df_filtered = df_clean[df_clean['intent'].isin(valid_intents)].copy()

print(f"Retained {len(df_filtered)} samples across {len(valid_intents)} distinct medical intents.")

# 3. Train-test split (80/20)
train_df, test_df = train_test_split(
    df_filtered,
    test_size=0.20,
    random_state=42,
    stratify=df_filtered['intent']
)

print(f"Training samples: {len(train_df)} | Testing samples: {len(test_df)}")

Retained 16406 samples across 15 distinct medical intents.
Training samples: 13124 | Testing samples: 3282


In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

# 1. Create an NLP pipeline: TF-IDF Vectorization + Machine Learning Classifier
intent_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), stop_words='english', max_features=10000)),
    ('classifier', LinearSVC(random_state=42))
])

# 2. Train the model
intent_pipeline.fit(train_df['query'], train_df['intent'])

# 3. Evaluate the model on the test dataset
y_pred = intent_pipeline.predict(test_df['query'])
print("=== Classification Report (Intent Accuracy) ===")
print(classification_report(test_df['intent'], y_pred))

=== Classification Report (Intent Accuracy) ===
                 precision    recall  f1-score   support

         causes       1.00      1.00      1.00       145
  complications       1.00      1.00      1.00         9
 considerations       0.11      0.04      0.06        47
exams and tests       0.99      1.00      1.00       131
      frequency       1.00      1.00      1.00       224
genetic changes       1.00      1.00      1.00       218
    information       0.95      0.98      0.97       907
    inheritance       1.00      1.00      1.00       289
        outlook       1.00      1.00      1.00        72
     prevention       1.00      1.00      1.00        42
       research       1.00      1.00      1.00        79
         stages       1.00      1.00      1.00        15
 susceptibility       1.00      1.00      1.00        65
       symptoms       1.00      1.00      1.00       550
      treatment       1.00      1.00      1.00       489

       accuracy                       

You can customize the tokenization process by providing a `tokenizer` function to `TfidfVectorizer`. For example, you can use `nltk` for more advanced tokenization. First, you might need to install and download `nltk` data:

In [18]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab') # Add this line to download the specific resource needed
from nltk.tokenize import word_tokenize

def custom_tokenizer(text):
    return word_tokenize(text.lower())

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Then, you can integrate this `custom_tokenizer` into your `TfidfVectorizer` within the `intent_pipeline`:

In [19]:
!pip install -q sentence-transformers

import numpy as np
import torch
import re
from sentence_transformers import SentenceTransformer, util

# ==========================================
# 1. INITIALIZE MODELS & DATABASE
# ==========================================
# Load Dense Semantic Embedder (SBERT)
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# Precompute embeddings with Focus, Intent, and Query to guide the search
# (Assumes train_df and intent_pipeline are already loaded in your environment)
corpus_texts = (
    train_df['focus_area'].fillna('').astype(str) + " " +
    train_df['intent'].fillna('').astype(str) + " " +
    train_df['query'].fillna('').astype(str)
).tolist()

print("Encoding database... this may take a moment.")
corpus_embeddings = embedder.encode(corpus_texts, convert_to_tensor=True, show_progress_bar=True)


# ==========================================
# 2. SENTIMENT & LEXICON CONFIGURATION
# ==========================================
# Define the Custom Lexicon for medical urgency
urgency_lexicon = {
    "pain": -1, "severe": -2, "emergency": -2, "bleeding": -2,
    "help": -1, "chronic": -1, "relieved": 1, "better": 1, "manageable": 1
}

def score_custom_lexicon(text, lexicon):
    """Tokenizes text and sums the scores of matching lexicon words."""
    words = text.lower().split()
    total_score = 0
    for word in words:
        clean_word = re.sub(r'[^\w\s]', '', word) # Strip punctuation
        if clean_word in lexicon:
            total_score += lexicon[clean_word]
    return total_score


# ==========================================
# 3. TEXT PROCESSING UTILITIES
# ==========================================
def clean_response(text):
    """Removes extra whitespace, newlines, and dataset boilerplate."""
    # Remove video prompts
    text = re.sub(r'\(Watch the video.*?\)', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\(To enlarge the video.*?\)', '', text, flags=re.IGNORECASE)

    # Remove graphic and glossary prompts (Fixes Q3)
    text = re.sub(r'See this graphic.*?(?=\.|$)\.?', '', text, flags=re.IGNORECASE)
    text = re.sub(r'See a glossary.*?(?=\.|$)\.?', '', text, flags=re.IGNORECASE)

    text = re.sub(r'\s+', ' ', text).strip()
    return text


# ==========================================
# 4. MAIN CHATBOT FUNCTION
# ==========================================
def get_chatbot_response(user_query, confidence_threshold=0.35):
    # Step A: Score the user's query using the custom lexicon
    urgency_score = score_custom_lexicon(user_query, urgency_lexicon)

    # Step B: Predict intent
    predicted_intent = intent_pipeline.predict([user_query])[0]

    # Step C: Dense search query formulation
    search_query = f"{predicted_intent} {user_query}"
    query_vec = embedder.encode(search_query, convert_to_tensor=True)

    # Step D: Semantic search in the database
    scores = util.cos_sim(query_vec, corpus_embeddings)[0]
    best_idx = torch.argmax(scores).item()
    best_score = scores[best_idx].item()

    # Step E: Guardrail confidence check
    if best_score < confidence_threshold:
        return {
            "intent": predicted_intent,
            "confidence": f"{best_score:.2f}",
            "urgency_score": urgency_score,
            "matched_focus": "N/A",
            "response": "I am not confident in answering this based on verified medical data. Please consult a licensed healthcare professional."
        }

    # Step F: Fetch, clean, and truncate the response
    matched_row = train_df.iloc[best_idx]
    raw_response = matched_row['response']

    clean_raw = clean_response(raw_response)
    final_answer = semantic_truncate(clean_raw, user_query, embedder, max_words=90)

    # Step G: Intervene if high urgency is detected
    if urgency_score < -1:
        final_answer = "This sounds like it might be urgent. Please consult a doctor immediately. " + final_answer

    return {
        "intent": predicted_intent,
        "confidence": f"{best_score:.2f}",
        "urgency_score": urgency_score,
        "matched_focus": matched_row['focus_area'],
        "response": final_answer
    }

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding database... this may take a moment.


Batches:   0%|          | 0/411 [00:00<?, ?it/s]

In [22]:
def semantic_truncate(text, query, embedder, max_words=90, min_words=30, truncation_stride=5):
    """Truncates text while trying to preserve semantic completeness based on a query.

    Args:
        text (str): The full text to be truncated.
        query (str): The user's query, used to guide truncation.
        embedder (SentenceTransformer): The sentence embedding model.
        max_words (int): The maximum number of words for the truncated text.
        min_words (int): The minimum number of words for the truncated text.
        truncation_stride (int): How many words to remove at a time.

    Returns:
        str: The semantically truncated text.
    """
    words = text.split()

    # If the text is already short enough, return it
    if len(words) <= max_words:
        return text

    query_embedding = embedder.encode(query, convert_to_tensor=True)
    best_truncation = text
    highest_similarity = -1

    # Iterate, truncating from the end
    for i in range(len(words) - min_words, max_words - 1, -truncation_stride):
        current_truncated_text = " ".join(words[:i])
        if not current_truncated_text:
            continue

        truncated_embedding = embedder.encode(current_truncated_text, convert_to_tensor=True)
        similarity = util.cos_sim(query_embedding, truncated_embedding).item()

        if similarity > highest_similarity:
            highest_similarity = similarity
            best_truncation = current_truncated_text

    # If the best truncation found is still too long, fall back to simple truncation
    if len(best_truncation.split()) > max_words:
        return " ".join(words[:max_words]) + "..."
    return best_truncation + "..."

In [25]:
# Test queries with urgent examples added
sample_queries = [
    "What is hepatitis C?",
    "I am in severe pain and need help with my chronic cough!", # Urgent test
    "How is glaucoma diagnosed by a doctor?",
    "Is there a cure for severe bleeding?" # Urgent test
]

for query in sample_queries:
    result = get_chatbot_response(query)
    print(f"\nUser Query: {query}")

    # Using .get() prevents errors if a key is missing
    print(f"Predicted Intent: {result.get('intent')} (Score: {result.get('confidence', 'N/A')})")
    print(f"Urgency Score: {result.get('urgency_score', 0)}")
    print(f"Matched Disease: {result.get('matched_focus')}")
    print(f"Medical Answer: {result.get('response')}")


User Query: What is hepatitis C?
Predicted Intent: information (Score: 0.92)
Urgency Score: 0
Matched Disease: Hepatitis C
Medical Answer: Your liver is the largest organ inside your body. It helps your body digest food, store energy, and remove poisons. Hepatitis is an inflammation of the liver. One type, hepatitis C, is caused by the hepatitis C virus (HCV). It usually spreads through contact with infected blood. It can also spread through sex with an infected person and from mother to baby during childbirth. Most people who are infected with hepatitis C don't have any symptoms for years. If you do get symptoms, you may feel as if you have...

User Query: I am in severe pain and need help with my chronic cough!
Predicted Intent: information (Score: 0.70)
Urgency Score: -5
Matched Disease: Cough
Medical Answer: This sounds like it might be urgent. Please consult a doctor immediately. The best way to treat a cough is to treat its cause. However, sometimes the cause is unknown. Other t

In [ ]:
res = get_chatbot_response("hepatitis?")
print("Focus:", res["matched_focus"])
print("Answer:", res["response"])

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import numpy as np

# 1. Get predictions for the test dataset
y_true = test_df['intent']
y_pred = intent_pipeline.predict(test_df['query'])

# 2. Calculate Overall Metrics (Weighted averages handle imbalanced medical data well)
accuracy = accuracy_score(y_true, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted')

print("=== OVERALL MODEL PERFORMANCE ===")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}\n")

# 3. Create a Confusion Matrix
# Because MedQuAD has many intents, a full matrix is too large.
# We will visualize the Top 10 most frequent intents.
top_intents = y_true.value_counts().nlargest(10).index
mask = y_true.isin(top_intents) & pd.Series(y_pred, index=y_true.index).isin(top_intents)

# Filter true and predicted labels for just the top 10
y_true_filtered = y_true[mask]
y_pred_filtered = y_pred[mask.values]

cm = confusion_matrix(y_true_filtered, y_pred_filtered, labels=top_intents)

# 4. Plot the Confusion Matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=top_intents, yticklabels=top_intents)
plt.title('Confusion Matrix: Top 10 Medical Intents', fontsize=16)
plt.ylabel('Actual User Intent', fontsize=12)
plt.xlabel('Predicted Intent by AI', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Interactive Chatbot Demo

Try asking some medical questions to the chatbot. Type `quit` or `exit` to end the session.

In [ ]:
while True:
    user_input = input("\nYour Query (type 'quit' or 'exit' to stop): ")
    if user_input.lower() in ['quit', 'exit']:
        print("Exiting chatbot demo. Goodbye!")
        break

    response = get_chatbot_response(user_input)
    print(f"Predicted Intent: {response['intent']} (Confidence: {response['confidence']})")
    if 'matched_focus' in response:
        print(f"Matched Focus Area: {response['matched_focus']}")
        print(f"Urgency Score: {result.get('urgency_score', 0)}")
    print(f"Chatbot Response: {response['response']}")

## Save and Load the Model

To persist your trained `intent_pipeline` model, you can use `joblib`. This allows you to save the model to a file and load it later without having to retrain it.

In [ ]:
import joblib

# Define a filename for your model
model_filename = 'intent_pipeline_model.joblib'

# Save the trained pipeline to a file
joblib.dump(intent_pipeline, model_filename)
print(f"Model saved to {model_filename}")

# To load the model back later (in a new session or script):
loaded_intent_pipeline = joblib.load(model_filename)
print(f"Model loaded from {model_filename}")

# You can now use the loaded model for predictions
# For example, let's predict the intent for a new query using the loaded model
new_query = "what are the symptoms of diabetes?"
predicted_intent_loaded = loaded_intent_pipeline.predict([new_query])[0]
print(f"\nNew query: '{new_query}'")
print(f"Predicted intent using loaded model: {predicted_intent_loaded}")

In [ ]:
# Create a dictionary mapping each intent to a list of its responses
response_map = df.groupby('intent')['response'].apply(list).to_dict()

print(f"Response map created with {len(response_map)} unique intents.")
# Display a sample of the response_map (e.g., for one intent)
if 'information' in response_map:
    print(f"Sample responses for 'information' intent: {response_map['information'][0][:100]}...")

In [ ]:
# @title
from google.colab import files

# Download both files directly to your local computer downloads folder
# Save your response dictionary using the .joblib extension
joblib.dump(response_map, 'response_map.joblib')
print("Response lookup map successfully saved to response_map.joblib")

In [ ]:
# @title
from google.colab import files

# Download both files to your local machine
files.download('intent_pipeline_model.joblib')
files.download('response_map.joblib')